# 👵 Nursing Home Fall Detection - YOLO11 Training (EXP-008: Scale-Up Model Capacity & High Resolution)

Notebook ini mengimplementasikan eksperimen **EXP-008**:
1. **Model Capacity Scale-Up:** Menggunakan `yolo11s.pt` (Small - 9.4M parameter), memberikan kapasitas representasi visual 3.5x lebih kaya dibanding model Nano (`yolo11n`).
2. **High-Resolution Input:** `imgsz=800` (naik dari 640) agar detail sendi dan orientasi tubuh manusia tetap tajam ketika berdiri, duduk, atau berada agak jauh dari kamera.
3. **Posture Disambiguation:** `cls=1.2` dan `label_smoothing=0.05` untuk mengatasi keraguan model antara kelas `transitional` vs `normal` vs `lying_on_ground`.
4. **Robust Lighting & Anti-Clutter:** Augmentasi intensif `hsv_v=0.6`, `hsv_s=0.7`, `bgr=0.2`, `erasing=0.4`, `scale=0.5` untuk ketahanan terhadap cahaya ekstrem dan perabotan kamar.

### 📌 Petunjuk Penggunaan di Kaggle:
1. Aktifkan GPU di menu kanan: **Settings -> Accelerator -> GPU T4 x1**.
2. Pastikan dataset sudah terhubung (Path: `/kaggle/input/datasets/rajarahmanaziiz/fall-detection-dataset`).
3. Jalankan seluruh cell berurutan dari atas ke bawah.

### 1. Install Ultralytics & Verifikasi GPU

In [ ]:
!pip install -q ultralytics

import torch
import yaml
from pathlib import Path
from ultralytics import YOLO

print('PyTorch Version :', torch.__version__)
print('CUDA Available  :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device Name     :', torch.cuda.get_device_name(0))


### 2. Deteksi Dataset Kaggle (Folder Langsung / Uncompressed)

In [ ]:
primary_dataset_path = Path('/kaggle/input/datasets/rajarahmanaziiz/fall-detection-dataset')
fallback_candidates = [
    primary_dataset_path,
    Path('/kaggle/input/fall-detection-dataset'),
    Path('/kaggle/input/urfall-yolo-dataset')
]

dataset_dir = None
for candidate in fallback_candidates:
    if candidate.exists():
        if (candidate / 'images').exists():
            dataset_dir = candidate
            break
        sub_images = list(candidate.glob('**/images'))
        if sub_images:
            dataset_dir = sub_images[0].parent
            break

if not dataset_dir:
    kaggle_input = Path('/kaggle/input')
    found_images = list(kaggle_input.glob('**/images'))
    if found_images:
        dataset_dir = found_images[0].parent

if dataset_dir:
    print(f'✅ Dataset ditemukan di: {dataset_dir}')
    print(f'   Train images : {len(list((dataset_dir / "images" / "train").glob("*")))} file')
    print(f'   Val images   : {len(list((dataset_dir / "images" / "val").glob("*")))} file')
    print(f'   Test images  : {len(list((dataset_dir / "images" / "test").glob("*")))} file')
else:
    raise FileNotFoundError('❌ Folder dataset tidak ditemukan di /kaggle/input!')


### 3. Buat file data.yaml

In [ ]:
work_dir = Path('/kaggle/working')
kaggle_yaml_path = work_dir / 'data.yaml'

data_yaml_content = {
    'path': str(dataset_dir.resolve()),
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'nc': 3,
    'names': {
        0: 'normal',
        1: 'transitional',
        2: 'lying_on_ground'
    }
}

with open(kaggle_yaml_path, 'w') as f:
    yaml.dump(data_yaml_content, f, default_flow_style=False)

print(f'✓ data.yaml siap di: {kaggle_yaml_path}')
print(open(kaggle_yaml_path).read())


### 4. Training Model EXP-008 (`yolo11s.pt` + `imgsz=800`)

In [ ]:
print('🚀 [EXP-008] Memulai Training YOLO11s (Resolution 800x800)...')

model = YOLO('yolo11s.pt')

results = model.train(
    data=str(kaggle_yaml_path),
    epochs=70,
    patience=25,
    batch=16,
    imgsz=800,
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    cos_lr=True,
    cls=1.2,
    label_smoothing=0.05,
    hsv_v=0.6,
    hsv_s=0.7,
    hsv_h=0.015,
    bgr=0.2,
    erasing=0.4,
    scale=0.5,
    flipud=0.0,
    fliplr=0.5,
    mosaic=1.0,
    close_mosaic=10,
    project='/kaggle/working/models',
    name='exp008_yolo11s_imgsz800',
    exist_ok=True,
    save=True
)

best_model_path = Path(results.save_dir) / 'weights' / 'best.pt'
print(f'\n✅ Training Selesai! Model terbaik tersimpan di: {best_model_path}')


### 5. Evaluasi Test Set Model Akhir

In [ ]:
print('\n🔍 Menjalankan evaluasi Test Set...')
eval_model = YOLO(str(best_model_path))
metrics = eval_model.val(data=str(kaggle_yaml_path), split='test', imgsz=800)

print('\n' + '=' * 55)
print('📊 HASIL EVALUASI TEST SET EXP-008 (YOLO11s @ 800x800):')
print('=' * 55)
print(f'• Test mAP50        : {metrics.box.map50 * 100:.2f}%')
print(f'• Test mAP50-95     : {metrics.box.map * 100:.2f}%')
print(f'• Test Precision    : {metrics.box.mp * 100:.2f}%')
print(f'• Test Recall       : {metrics.box.mr * 100:.2f}%')
print('-' * 55)

if hasattr(metrics.box, 'maps') and len(metrics.box.maps) == 3:
    class_names = ['normal', 'transitional', 'lying_on_ground']
    for i, name in enumerate(class_names):
        print(f'  - {name:<16}: mAP50-95 = {metrics.box.maps[i] * 100:.2f}%')

print('=' * 55)
print(f'\n📁 Model bobot terbaik siap diunduh:')
print(f'   {best_model_path}')
